# DiGress GSG Candidate Generation and Validation

This notebook trains a DiGress discrete graph diffusion model on 154 known genus-9 GSG examples, generates candidate graphs, filters them by structural checks, and validates the structurally valid candidates using ChipFiring.jl.

Pipeline:
1. Load known genus-9 GSGs
2. Convert graphs to PyTorch Geometric format
3. Train DiGress
4. Generate 50 candidate graphs
5. Filter by structural checks: V=16, E=24, trivalent, connected
6. Validate candidates with ChipFiring.jl


In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install pytorch-lightning==2.0.4
!pip install hydra-core==1.3.2 omegaconf==2.3.0
!pip install wandb==0.15.4 torchmetrics==0.11.4
!pip install networkx matplotlib
!git clone https://github.com/cvignac/DiGress.git
import sys
sys.path.insert(0, '/content/DiGress')
print("Done!")

In [ ]:
from google.colab import files
print("Upload known_gsgs.txt:")
uploaded = files.upload()

In [ ]:
from torch_geometric.data import Data

def multigraph_to_pyg(G):
    nodes = sorted(G.nodes())
    n = len(nodes)
    X = torch.ones(n, NUM_NODE_TYPES, dtype=torch.float)
    y = torch.zeros([1, 0]).float()
    edge_indices = []
    edge_attrs = []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            mult = G.number_of_edges(nodes[i], nodes[j])
            if mult > 0:
                edge_indices.append([i, j])
                attr = [0.0] * NUM_EDGE_TYPES
                attr[min(mult, NUM_EDGE_TYPES - 1)] = 1.0
                edge_attrs.append(attr)
    if edge_indices:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, NUM_EDGE_TYPES), dtype=torch.float)
    return Data(x=X, edge_index=edge_index, edge_attr=edge_attr, y=y, n_nodes=n * torch.ones(1, dtype=torch.long))

pyg_graphs = [multigraph_to_pyg(G) for G in graphs]
print(f"Converted {len(pyg_graphs)} graphs to PyG format")

In [ ]:
import os

random.seed(42)
indices = list(range(len(pyg_graphs)))
random.shuffle(indices)
train_size = int(0.7 * len(indices))
val_size = int(0.15 * len(indices))
train_data = [pyg_graphs[i] for i in indices[:train_size]]
val_data = [pyg_graphs[i] for i in indices[train_size:train_size + val_size]]
test_data = [pyg_graphs[i] for i in indices[train_size + val_size:]]

os.makedirs("/content/data/gsg/raw", exist_ok=True)
os.makedirs("/content/data/gsg/processed", exist_ok=True)
torch.save(train_data, "/content/data/gsg/raw/train.pt")
torch.save(val_data, "/content/data/gsg/raw/val.pt")
torch.save(test_data, "/content/data/gsg/raw/test.pt")
print(f"Split: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

In [ ]:
_original_torch_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_torch_load(*args, **kwargs)
torch.load = _safe_load

from torch_geometric.data import InMemoryDataset
from src.datasets.abstract_dataset import AbstractDataModule, AbstractDatasetInfos
from src.diffusion.distributions import DistributionNodes

class GSGDataset(InMemoryDataset):
    def __init__(self, split, root="/content/data/gsg", transform=None, pre_transform=None, pre_filter=None):
        self.split = split
        super().__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])
    @property
    def raw_file_names(self):
        return ['train.pt', 'val.pt', 'test.pt']
    @property
    def processed_file_names(self):
        return [self.split + '.pt']
    def download(self):
        pass
    def process(self):
        file_idx = {'train': 0, 'val': 1, 'test': 2}
        raw_dataset = torch.load(self.raw_paths[file_idx[self.split]])
        torch.save(self.collate(raw_dataset), self.processed_paths[0])

class GSGDataModule(AbstractDataModule):
    def __init__(self, cfg):
        self.cfg = cfg
        datasets = {'train': GSGDataset(split='train'), 'val': GSGDataset(split='val'), 'test': GSGDataset(split='test')}
        super().__init__(cfg, datasets)
        self.inner = self.train_dataset
    def __getitem__(self, item):
        return self.inner[item]

class GSGDatasetInfos(AbstractDatasetInfos):
    def __init__(self, datamodule, dataset_config):
        self.datamodule = datamodule
        self.name = 'gsg_graphs'
        self.n_nodes = self.datamodule.node_counts()
        self.node_types = torch.tensor([1.0])
        self.edge_types = self.datamodule.edge_counts()
        super().complete_infos(self.n_nodes, self.node_types)

print(f"Train dataset size: {len(GSGDataset(split='train'))}")

In [ ]:
import os, sys
os.chdir('/content/DiGress/src')
if '/content/DiGress/src' not in sys.path:
    sys.path.insert(0, '/content/DiGress/src')

from omegaconf import OmegaConf
from src.diffusion_model_discrete import DiscreteDenoisingDiffusion
from src.diffusion.extra_features import DummyExtraFeatures, ExtraFeatures
from metrics.abstract_metrics import TrainAbstractMetricsDiscrete
import pytorch_lightning as pl

cfg = OmegaConf.create({
    'general': {'name': 'gsg_generation', 'wandb': 'disabled', 'gpus': 1, 'resume': None, 'test_only': None,
        'check_val_every_n_epochs': 10, 'sample_every_val': 10, 'val_check_interval': None,
        'samples_to_generate': 10, 'samples_to_save': 10, 'chains_to_save': 0, 'log_every_steps': 50,
        'number_chain_steps': 50, 'final_model_samples_to_generate': 50,
        'final_model_samples_to_save': 10, 'final_model_chains_to_save': 0, 'evaluate_all_checkpoints': False},
    'train': {'batch_size': 16, 'num_workers': 0, 'lr': 2e-4, 'weight_decay': 1e-12, 'n_epochs': 300,
        'save_model': True, 'ema_decay': 0.999, 'optimizer': 'adamw', 'clip_grad': None, 'progress_bar': True, 'seed': 42},
    'model': {'type': 'discrete', 'model': 'graph_tf', 'transition': 'marginal', 'diffusion_steps': 500,
        'diffusion_noise_schedule': 'cosine', 'n_layers': 5, 'extra_features': 'cycles',
        'hidden_mlp_dims': {'X': 128, 'E': 64, 'y': 64},
        'hidden_dims': {'dx': 128, 'de': 64, 'dy': 64, 'n_head': 4, 'dim_ffX': 128, 'dim_ffE': 64, 'dim_ffy': 64},
        'lambda_train': [1.0, 0.0]},
    'dataset': {'name': 'gsg', 'datadir': '/content/data/gsg', 'pin_memory': False, 'remove_h': False}
})

datamodule = GSGDataModule(cfg)
dataset_infos = GSGDatasetInfos(datamodule, cfg.dataset)
extra_features = ExtraFeatures(extra_features_type='cycles', dataset_info=dataset_infos)
domain_features = DummyExtraFeatures()
dataset_infos.compute_input_output_dims(datamodule, extra_features, domain_features)

train_metrics = TrainAbstractMetricsDiscrete()

class DummySamplingMetrics:
    def reset(self): pass
    def forward(self, *a, **k): pass
    def __call__(self, *a, **k): pass

model = DiscreteDenoisingDiffusion(cfg=cfg, dataset_infos=dataset_infos, train_metrics=train_metrics,
    sampling_metrics=DummySamplingMetrics(), visualization_tools=None,
    extra_features=extra_features, domain_features=domain_features)

trainer = pl.Trainer(max_epochs=cfg.train.n_epochs, accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1, log_every_n_steps=10, enable_checkpointing=True)

print("Training...")
trainer.fit(model, datamodule)
print("Done!")

In [ ]:
model.eval()
num_to_generate = 50
print(f"Generating {num_to_generate} candidates...")

with torch.no_grad():
    molecule_list = model.sample_batch(batch_id=0, batch_size=num_to_generate,
        keep_chain=0, number_chain_steps=1, save_final=0, num_nodes=NUM_NODES)

generated_graphs = []
for atom_types, edge_types in molecule_list:
    n = atom_types.shape[0]
    G = nx.MultiGraph()
    G.add_nodes_from(range(1, n + 1))
    for row in range(n):
        for col in range(row + 1, n):
            mult = int(edge_types[row, col].item())
            for _ in range(mult):
                G.add_edge(row + 1, col + 1)
    generated_graphs.append(G)

print(f"Generated {len(generated_graphs)} graphs")

In [ ]:
def graph_to_edge_list(G):
    V, E = G.number_of_nodes(), G.number_of_edges()
    edges = sorted(G.edges())
    return f"Graph(V={V}, E={E}, Edges=[{', '.join(f'({u}, {v})' for u, v in edges)}])"

valid_count = 0
valid_graphs = []
for G in generated_graphs:
    is_valid = (G.number_of_nodes() == 16 and G.number_of_edges() == 24
                and all(G.degree(v) == 3 for v in G.nodes()) and nx.is_connected(G))
    if is_valid:
        valid_count += 1
        valid_graphs.append(G)

print(f"Structurally valid: {valid_count}/{len(generated_graphs)} ({100*valid_count/len(generated_graphs):.1f}%)")

with open("digress_candidates.txt", "w") as f:
    f.write(f"# {len(valid_graphs)} candidates from DiGress trained on {len(graphs)} genus 9 GSGs\n\n")
    for i, G in enumerate(valid_graphs):
        f.write(f'Candidate #{i+1}: "{graph_to_edge_list(G)}"\n')

files.download("digress_candidates.txt")
print(f"Saved and downloaded {len(valid_graphs)} candidates")

In [ ]:
## ChipFiring Validation

The previous section only checks structural validity: `V=16`, `E=24`, trivalent, and connected.

This section checks whether the structurally valid DiGress candidates are actual gonality-savings graphs using the ChipFiring validation code.


In [ ]:
!curl -fsSL https://install.julialang.org | sh -s -- --yes

import os
os.environ["PATH"] = os.path.expanduser("~/.juliaup/bin") + ":" + os.environ["PATH"]

!julia -e 'using Pkg; Pkg.add(["Graphs", "TreeWidthSolver"]); Pkg.add(url="https://github.com/vincentxwang/ChipFiring.jl")'
!pip install juliacall

print("Julia + ChipFiring + juliacall installed.")


In [ ]:
from google.colab import files

print("Upload compute_gonality.jl from the chip-firing-rl repo:")
uploaded = files.upload()


In [ ]:
from juliacall import Main as jl

jl.include("compute_gonality.jl")

to_julia_matrix = jl.seval(
    "py_list -> [Int64(py_list[i][j]) for i in 1:length(py_list), j in 1:length(py_list[1])]"
)

def multigraph_to_matrix(G):
    nodes = sorted(G.nodes())
    n = len(nodes)
    matrix = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(G.number_of_edges(nodes[i], nodes[j]))
        matrix.append(row)
    return matrix

def validate_gsg(G):
    mat = multigraph_to_matrix(G)
    jl_matrix = to_julia_matrix(mat)
    g = jl.ChipFiringGraph(jl_matrix)

    gon = int(jl.compute_gonality(g))
    genus = int(jl.compute_genus(g))

    if gon <= 4 or genus <= 5 or G.number_of_nodes() <= 5:
        return gon, gon, False

    ugon_approx = gon
    rank = gon

    while rank > 0:
        result = int(
            jl.compute_gonality(
                jl.subdivide(g, 2),
                min_d=rank,
                max_d=rank
            )
        )

        if result == -1:
            ugon_approx = rank + 1
            break

        rank -= 1

    return gon, ugon_approx, gon != ugon_approx

validation_results = []

print(f"Validating {len(valid_graphs)} structurally valid DiGress candidates...")
print("-" * 50)

for i, G in enumerate(valid_graphs):
    print(f"Candidate #{i+1}: computing...", end=" ", flush=True)

    try:
        gon, ugon, is_gsg = validate_gsg(G)
        status = "GSG" if is_gsg else "not GSG"

        validation_results.append({
            "candidate": i + 1,
            "gon": gon,
            "ugon_approx": ugon,
            "is_gsg": is_gsg,
        })

        print(f"gon={gon}, ugon_approx={ugon} → {status}")

    except Exception as e:
        validation_results.append({
            "candidate": i + 1,
            "error": str(e),
        })

        print(f"error: {e}")

print("-" * 50)
print("Validation complete.")
